In [ ]:
# Robust loader handling both old "batch" and "flattened" merged formats

import glob
import pickle
import pandas as pd

files = sorted(glob.glob("data_batches/merged_*.pkl") + glob.glob("data_batches/batch_*.pkl"))

rows = []
for fp in files:
    with open(fp, "rb") as f:
        data = pickle.load(f)
    
    # Data may be:
    # 1) A dict with "params" (N×7 array) and "results" (list of dicts)
    # 2) A list whose elements are either:
    #    a) batch-dicts with "params"/"results" (old-merged)
    #    b) single-point dicts with "params" (length-7 array) and "result"/"results" (dict)
    
    # Helper to flatten a batch-dict
    def flatten_batch(batch):
        ps = batch["params"]
        rs = batch["results"]
        for p, r in zip(ps, rs):
            yield p, r
    
    # Helper to flatten a single-point dict
    def flatten_point(entry):
        p = entry["params"]
        # entry may use "result" or "results"
        r = entry.get("result", entry.get("results"))
        if isinstance(r, list):  # sometimes "results" is a list of 1 dict
            r = r[0]
        yield p, r
    
    # Now dispatch
    if isinstance(data, dict) and "params" in data and "results" in data:
        # Single batch file
        for p, r in flatten_batch(data):
            row = {
                "m_A":      p[0],
                "m_phi":    p[1],
                "sin_ba":   p[2],
                "tan_beta": p[3],
                "lambda_6": p[4],
                "lambda_7": p[5],
                "m12_2":    p[6],
                **r
            }
            rows.append(row)
    
    elif isinstance(data, list):
        # Merged file: list of entries
        for entry in data:
            if isinstance(entry, dict):
                if "params" in entry and "results" in entry:
                    # This entry is a full batch
                    for p, r in flatten_batch(entry):
                        row = {
                            "m_A":      p[0],
                            "m_phi":    p[1],
                            "sin_ba":   p[2],
                            "tan_beta": p[3],
                            "lambda_6": p[4],
                            "lambda_7": p[5],
                            "m12_2":    p[6],
                            **r
                        }
                        rows.append(row)
                elif "params" in entry:
                    # Single-point entry
                    for p, r in flatten_point(entry):
                        row = {
                            "m_A":      p[0],
                            "m_phi":    p[1],
                            "sin_ba":   p[2],
                            "tan_beta": p[3],
                            "lambda_6": p[4],
                            "lambda_7": p[5],
                            "m12_2":    p[6],
                            **r
                        }
                        rows.append(row)
                else:
                    raise ValueError(f"Unexpected dict keys in entry from {fp}: {entry.keys()}")
            else:
                raise ValueError(f"Unexpected entry type in merged list from {fp}: {type(entry)}")
    else:
        raise ValueError(f"Unknown data structure in file {fp}: {type(data)}")

# 3) Build DataFrame and save to Parquet (or fallback to CSV)
df = pd.DataFrame(rows)

try:
    df.to_parquet("data_train/dataset_full.parquet", index=False)
    print(f"Saved merged dataset with {len(df)} rows to dataset_full.parquet")
except Exception as e:
    print("Failed to write Parquet:", e)
    df.to_csv("data_train/dataset_full.csv", index=False)
    print("Saved merged dataset to dataset_full.csv")


Saved merged dataset with 570400 rows to dataset_full.parquet


In [22]:
df.dropna(inplace=True)

# Training

In [ ]:
print(
    sum(df["positivity_ok"]),
    sum(df["perturbativity_ok"]),
    sum(df["unitarity_ok"]) )

285416.0 2.0 16.0


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import qmc
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

# 1) Load dataset
df = pd.read_parquet("data_train/dataset_full.parquet")

# 2) Derive perturbativity score
lambda_cols = ["lambda1", "lambda2", "lambda3", "lambda4", "lambda5", "lambda6", "lambda7"]
df["perturb_score"] = (df[lambda_cols]**2).sum(axis=1)

# 3) Descriptive statistics by class
desc_pos = df.groupby("perturbativity_ok")[lambda_cols + ["perturb_score"]].describe()
print("Descriptive statistics for perturbativity_ok:")
print(desc_pos)

# 4) Histograms of perturb_score by class
plt.figure()
for cls in [0, 1]:
    subset = df[df["perturbativity_ok"] == cls]
    plt.hist(subset["perturb_score"], bins=50, alpha=0.5, label=f"class {cls}")
plt.legend()
plt.xlabel("Perturbativity score")
plt.ylabel("Count")
plt.title("Distribution of perturbativity score by class")

# 5) Prepare features and labels
X = df[["m_A", "m_phi", "m_12", "tan_beta", "sin_ba", "lambda_6", "lambda_7", "perturb_score"]]
y = df["perturbativity_ok"]

# 6) Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

# 7) Standardize
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 8) Model 1: Logistic Regression
logreg = LogisticRegression(max_iter=1000)
logreg_scores = cross_val_score(logreg, X_train_scaled, y_train, cv=5, scoring="accuracy")
print(f"Logistic Regression CV accuracy: {logreg_scores.mean():.3f}")

# 9) Model 2: Random Forest
rf = RandomForestClassifier(n_estimators=100)
rf_scores = cross_val_score(rf, X_train, y_train, cv=5, scoring="accuracy")
print(f"Random Forest CV accuracy: {rf_scores.mean():.3f}")

# 10) Hyperparameter tuning for RF
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 5, 10],
    "min_samples_split": [2, 5]
}
grid = GridSearchCV(rf, param_grid, cv=3, scoring="accuracy", n_jobs=-1)
grid.fit(X_train, y_train)
print("Best RF params:", grid.best_params_)
print(f"Tuned RF CV accuracy: {grid.best_score_:.3f}")


KeyError: "['lambda_1', 'lambda_2', 'lambda_3', 'lambda_4', 'lambda_5'] not in index"